# abhyasa

> A learning harness that makes you think: active recall, a learner model and a concept graph in your vault.

abhyasa (अभ्यास) means steady practice. It is a tutor that asks you questions and makes you do the thinking.
It never gives the answer first. You answer in text or by voice, and you can think aloud.
The tutor grades your reasoning, not only the result.
A learner model keeps the depth of each concept that you know, and the date of its next review.

In [ ]:
#| hide
from abhyasa.store import *

## How it works

Each question goes through one loop:

1. Pick. Code chooses the next concept and the depth: a due review first, then the frontier. The app shows the reason.
2. Ask. A model writes one question at that depth, with a hidden model answer and a rubric.
3. Struggle. You answer. A wrong try gets the first error and a nudge, but no fix. You can ask for hints, one rung at a time.
4. Grade. The model grades the answer and the think-aloud. Code grades choice questions and code exercises.
5. Update. Correctness and confidence move the floor, the ceiling and the next review date of the concept.
6. Teach. After a miss, the tutor teaches the smallest step that fixes the first error. Then it asks a check question.

| term | meaning |
|---|---|
| concept | one idea that you can be asked about |
| depth | 0 unseen, 1 recognise, 2 recall, 3 explain, 4 apply, 5 transfer |
| floor | the highest depth that you pass reliably |
| ceiling | the lowest depth that you failed |
| attempt | one answer, kept as a vault document |
| session | one sitting of questions, with a summary |
| hint | one rung of a ladder of six, from the least to the most revealing |
| lesson | the smallest step that the tutor teaches after a miss |

## Install and run

Install the package and its dependencies:

```sh
uv sync
```

Start the web app on <http://127.0.0.1:5077>:

```sh
uv run abhyasa
```

The app listens on the loopback address only. These variables change the defaults:

| variable | what it sets | default |
|---|---|---|
| `ABHYASA_VAULT` | the vault file | the vault of vishalakshi |
| `ABHYASA_MODEL` | the rishi model for questions, grades, hints and lessons | `claude/sonnet` |
| `ABHYASA_STT` | the speech-to-text engine: `gemma`, `whisper`, `parakeet`, `openai` or `mlx:<repo>`. It is stronger than the Settings page. | the Settings page, then `gemma` |

## Surfaces

All surfaces use the same tutor and the same data.

### Web app

The web app is made from blocks under `abhyasa/blocks`:

| block | what it does |
|---|---|
| `today` | the home page: what is due and what to do now |
| `session` | one sitting with the tutor: one question at a time, with hints, confidence and "I don't know" |
| `voice` | a mic button for each text box; it listens until you stop it, and the text comes in at each pause |
| `settings` | the speech model: pick, look up and get one; the OpenAI key; the tutor model |
| `concepts` | the list of concepts, and one page for each concept with its state |
| `graph` | the concept graph: a color for each floor, a ring when a review is due, arrows for prereqs |
| `search` | one search over the `learning` shelf and the other shelves of your vault |
| `add` | add concepts from a text, a vault document or vedicreader |

### ramabana

The `/learn` extension gives the tutor in the terminal, beside your code.
To turn it on, make the file `~/.config/ramabana/extensions/abhyasa.py` with this line:

```python
from abhyasa.rama import setup
```

### Notebooks

In leela or Jupyter, `learn` shows a question card. `exercise` gives a code exercise that its tests check in the kernel.

```python
from abhyasa.nb import learn, exercise
learn('reciprocal-rank-fusion')
ex = exercise('reciprocal-rank-fusion')
```

## Where the data is

All data is on the `learning` shelf of the vishalakshi vault.
Each concept, attempt and session is one vault document, so other tools can search it.
Tests use a vault in a temporary folder and never open your real vault.

## Example

This example runs offline. `temp_learning` makes a shelf in a temporary vault, and `Fake` replaces the model.

In [ ]:
from abhyasa.store import temp_learning
from abhyasa.llm import Fake
from abhyasa.tutor import Tutor

lr = temp_learning()
lr.put('Ranking', 'Order items by a score.', 'ml')
fake = Fake({'You grade one answer': dict(correct=True, grade=8, first_error='', misconception='', feedback='Yes. Only the order counts.')},
            dflt=dict(question='Why can you compare ranks from two systems, but not their scores?', answer='Ranks share one scale.', rubric=['ranks share one scale']))
t = Tutor(lr, fake, context=False)

Ask the first question. `why` gives the reason for the choice. A new concept starts at depth 3 (explain).

In [ ]:
q = t.next()
assert q.why == 'next on the frontier: its prereqs are in place'
assert q.depth == 3
print(q.question)

Answer with high confidence. A `solid` pass at depth 3 raises the floor to 3.

In [ ]:
r = t.answer(q.id, 'A rank is a position, and positions use the same scale in each list.', confidence=80)
assert r.outcome == 'solid'
assert lr.concept('ranking').floor == 3